In [ ]:

import pandas as pd

# ---------------------------------------------------------------
# STEP 1: Load and profile the raw file
# ---------------------------------------------------------------
df = pd.read_csv("../data/meridian_churn_data.csv")

print("Shape:", df.shape)
print("Duplicate customer_id count:", df["customer_id"].duplicated().sum())
print("Fully blank rows:", df.isna().all(axis=1).sum())
print("\nMissing values per column:\n", df.isna().sum())


Shape: (1400, 14)
Duplicate customer_id count: 0
Fully blank rows: 0

Missing values per column:
 customer_id                          0
segment                              0
plan_tier                            0
monthly_price                        0
tenure_months                        0
signup_channel                       0
support_tickets_last_90d             0
avg_support_response_hours           0
product_usage_score                  0
num_users_licensed                   0
num_users_active                     0
had_price_increase_last_quarter      0
nps_score                          380
churned                              0
dtype: int64


In [ ]:
# STEP 2: Check each numeric column's range for implausible values
# ---------------------------------------------------------------
print("\nmonthly_price range:", df["monthly_price"].min(), "to", df["monthly_price"].max())
neg_price_rows = df[df["monthly_price"] < 0]
print("Negative price rows found:", len(neg_price_rows))
print(neg_price_rows[["customer_id", "segment", "plan_tier", "monthly_price"]])


monthly_price range: -517.9 to 573.7
Negative price rows found: 3
    customer_id     segment   plan_tier  monthly_price
156     C100156  Enterprise  Enterprise        -517.90
478     C100478         SMB       Basic         -46.62
609     C100609         SMB       Basic         -55.22


In [ ]:
# STEP 3: Diagnose the negative prices
# Compare each negative value's absolute size to its plan tier's
# typical (median) price to confirm it's a sign-flip, not a real
# credit/refund.
# ---------------------------------------------------------------
for idx, row in neg_price_rows.iterrows():
    plan_median = df.loc[df["plan_tier"] == row["plan_tier"], "monthly_price"].abs().median()
    print(
        f"{row['customer_id']} ({row['plan_tier']}): "
        f"raw={row['monthly_price']}, abs={abs(row['monthly_price'])}, "
        f"plan median={plan_median:.2f}"
    )

C100156 (Enterprise): raw=-517.9, abs=517.9, plan median=510.90
C100478 (Basic): raw=-46.62, abs=46.62, plan median=50.48
C100609 (Basic): raw=-55.22, abs=55.22, plan median=50.48


In [ ]:
# STEP 4: Correct the sign (keep the rows, don't drop them)
# ---------------------------------------------------------------
df["monthly_price"] = df["monthly_price"].abs()
assert (df["monthly_price"] < 0).sum() == 0, "Negative prices remain after cleaning"
print("monthly_price range:", df["monthly_price"].min(), "to", df["monthly_price"].max())

monthly_price range: 44.13 to 573.7


In [ ]:
# STEP 5: Check nps_score missingness
# ---------------------------------------------------------------
missing_nps = df["nps_score"].isna().sum()
print(f"\nnps_score missing: {missing_nps} of {len(df)} ({missing_nps/len(df):.1%})")


nps_score missing: 380 of 1400 (27.1%)


In [ ]:
# STEP 6: Flag non-response instead of imputing a fake score
# ---------------------------------------------------------------
df["nps_responded"] = df["nps_score"].notna()

In [ ]:
# STEP 7: Add derived column — seat utilization
# ---------------------------------------------------------------
df["seat_utilization"] = df["num_users_active"] / df["num_users_licensed"]

In [ ]:
# STEP 8: Re-check logical consistency after cleaning
checks = {
    "active seats > licensed seats": (df["num_users_active"] > df["num_users_licensed"]).sum(),
    "negative tenure_months": (df["tenure_months"] < 0).sum(),
    "usage_score out of 0-100 range": (~df["product_usage_score"].between(0, 100)).sum(),
    "nps_score out of -100..100 range": (~df["nps_score"].between(-100, 100) & df["nps_score"].notna()).sum(),
    "negative monthly_price remaining": (df["monthly_price"] < 0).sum(),
}
print("\nPost-clean consistency checks (should all be 0):")
for check, count in checks.items():
    print(f"  {check}: {count}")



Post-clean consistency checks (should all be 0):
  active seats > licensed seats: 0
  negative tenure_months: 0
  usage_score out of 0-100 range: 0
  nps_score out of -100..100 range: 0
  negative monthly_price remaining: 0
